<a href="https://colab.research.google.com/github/ANURAGYADAV008/ANURAGYADAV008/blob/main/qwen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install -U transformers peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.1/12.1 MB 79.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 64.4 MB/s eta 0:00:00


In [2]:
!pip install trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 992.6/992.6 kB 21.5 MB/s eta 0:00:00


In [3]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from datasets import load_dataset
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

assert torch.cuda.is_available(), "In Colab, enable Runtime → Change runtime type → T4 GPU."

print(torch.cuda.get_device_name(0))
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

MODEL_ID = "Qwen/Qwen3-4B"
OUTPUT_DIR = "qwen3-4b-lora"
MAX_LENGTH = 512       # T4-safe
NUM_TRAIN_SAMPLES = 2000  # Set None later to use the complete dataset.
SEED = 42

torch.manual_seed(SEED)

Tesla T4
VRAM: 14.6 GB


In [4]:
login()

NameError: name 'login' is not defined

**Load Qwen3-4B in 4-bit QLoRA mode**

In [5]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

model.config.use_cache = False
model.config.pad_token_id = tokenizer.pad_token_id

print("Model loaded.")

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded.


**Add LoRA adapters**

In [6]:
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145


**Load the reasoning and conversation datasets**

In [7]:
dataset = load_dataset(
    "unsloth/OpenMathReasoning-mini",
    split="cot",
)

if NUM_TRAIN_SAMPLES is not None:
    dataset = dataset.shuffle(seed=SEED).select(
        range(min(NUM_TRAIN_SAMPLES, len(dataset)))
    )

print(dataset)
print(dataset[0])

# print("Reasoning dataset:", reasoning_raw)
# print("Chat dataset:", chat_raw)

# print("\nReasoning example:")
# print(reasoning_raw[0])

# print("\nChat example:")
# print(chat_raw[0])

README.md:   0%|          | 0.00/603 [00:00<?, ?B/s]

data/cot-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  106MB            

data/cot-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating cot split:   0%|          | 0/19252 [00:00<?, ? examples/s]

Dataset({
    features: ['expected_answer', 'problem_type', 'problem_source', 'generation_model', 'pass_rate_72b_tir', 'problem', 'generated_solution', 'inference_mode'],
    num_rows: 2000
})
{'expected_answer': '\\(-\\ln(x) \\cos(x) + Ci(x) + C\\)', 'problem_type': 'has_answer_extracted', 'problem_source': 'aops_c7_college_math', 'generation_model': 'DeepSeek-R1', 'pass_rate_72b_tir': '0.96875', 'problem': 'How do you integrate $\\int \\ln{x}\\sin{x}\\ dx$?', 'generated_solution': "<think>\nOkay, so I need to integrate the function ln(x) times sin(x) dx. Hmm, let's see. I remember that when you have an integral that's a product of two functions, integration by parts might be useful. The formula for integration by parts is ∫u dv = uv - ∫v du. Right. \n\nFirst, I should decide which part of the integrand to set as u and which as dv. Typically, you want to choose u as the part that becomes simpler when you differentiate it. So here we have ln(x) and sin(x). The derivative of ln(x) is 1/

**Convert the reasoning dataset to chat messages**

In [8]:
def create_messages(example):
    return {
        "messages": [
            {"role": "user", "content": example["problem"]},
            {"role": "assistant", "content": example["generated_solution"]},
        ]
    }

dataset = dataset.map(
    create_messages,
    remove_columns=dataset.column_names,
)

print(dataset[0])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

{'messages': [{'role': 'user', 'content': 'How do you integrate $\\int \\ln{x}\\sin{x}\\ dx$?'}, {'role': 'assistant', 'content': "<think>\nOkay, so I need to integrate the function ln(x) times sin(x) dx. Hmm, let's see. I remember that when you have an integral that's a product of two functions, integration by parts might be useful. The formula for integration by parts is ∫u dv = uv - ∫v du. Right. \n\nFirst, I should decide which part of the integrand to set as u and which as dv. Typically, you want to choose u as the part that becomes simpler when you differentiate it. So here we have ln(x) and sin(x). The derivative of ln(x) is 1/x, which is simpler, and the integral of sin(x) is -cos(x), which doesn't get more complicated. So maybe let u = ln(x) and dv = sin(x) dx.\n\nLet me write that down:\n\nLet u = ln(x) ⇒ du = (1/x) dx\ndv = sin(x) dx ⇒ v = -cos(x)\n\nApplying the integration by parts formula:\n\n∫ ln(x) sin(x) dx = uv - ∫ v du\n= -ln(x) cos(x) - ∫ (-cos(x))(1/x) dx\n= -ln(x)

**Convert FineTome’s ShareGPT format to chat messages**

In [9]:
def format_conversation(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        enable_thinking=True,
    )
    return {"text": text}

dataset = dataset.map(
    format_conversation,
    remove_columns=dataset.column_names,
)

print(dataset[0]["text"][:2000])

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

<|im_start|>user
How do you integrate $\int \ln{x}\sin{x}\ dx$?<|im_end|>
<|im_start|>assistant
<think>
Okay, so I need to integrate the function ln(x) times sin(x) dx. Hmm, let's see. I remember that when you have an integral that's a product of two functions, integration by parts might be useful. The formula for integration by parts is ∫u dv = uv - ∫v du. Right. 

First, I should decide which part of the integrand to set as u and which as dv. Typically, you want to choose u as the part that becomes simpler when you differentiate it. So here we have ln(x) and sin(x). The derivative of ln(x) is 1/x, which is simpler, and the integral of sin(x) is -cos(x), which doesn't get more complicated. So maybe let u = ln(x) and dv = sin(x) dx.

Let me write that down:

Let u = ln(x) ⇒ du = (1/x) dx
dv = sin(x) dx ⇒ v = -cos(x)

Applying the integration by parts formula:

∫ ln(x) sin(x) dx = uv - ∫ v du
= -ln(x) cos(x) - ∫ (-cos(x))(1/x) dx
= -ln(x) cos(x) + ∫ (cos(x)/x) dx

Wait, now I have to co

In [10]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dataset.column_names,
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

print(tokenized_dataset)

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 2000
})


**Configure training for the T4 GPU**

In [11]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    learning_rate=2e-4,
    num_train_epochs=1,

    # Keep these OFF on a Tesla T4.
    fp16=False,
    bf16=False,

    gradient_checkpointing=True,

    logging_steps=5,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    report_to="none",
    seed=SEED,
)

**Create trainer**

In [12]:
print(f"model.config.output_hidden_states before SFTTrainer: {model.config.output_hidden_states}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator,
)

model.config.output_hidden_states before SFTTrainer: False


In [13]:
trainer.args.max_steps = -1
trainer.args.num_train_epochs = 1

results = trainer.train()

print(results.metrics)

Step,Training Loss
5,0.725669
10,0.555723
15,0.528114
20,0.514981
25,0.480321
30,0.503029
35,0.476964
40,0.473794
45,0.468402
50,0.448547


{'train_runtime': 4186.9422, 'train_samples_per_second': 0.478, 'train_steps_per_second': 0.06, 'total_flos': 2.252723453952e+16, 'train_loss': 0.4660320730209351, 'epoch': 1.0}


In [14]:
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"Saved to: {OUTPUT_DIR}")

Saved to: qwen3-4b-lora


In [15]:
model.eval()

messages = [
    {"role": "user", "content": "Solve (x + 2)^2 = 0."}
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.6,
        top_p=0.95,
        top_k=20,
        pad_token_id=tokenizer.eos_token_id,
    )

answer_tokens = output[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(answer_tokens, skip_special_tokens=False))

<think>
Okay, let's see. I need to solve the equation (x + 2)^2 = 0. Hmm, how do I approach this? Well, I remember that when you have a square equals zero, the inside of the square must also be zero. Because if something squared is zero, then that something has to be zero. So, if (x + 2)^2 is zero, then x + 2 must be zero. Let me write that down.

Starting with the original equation:
(x + 2)^2 = 0

Since the square of a number is zero only when the number itself is zero, I can take the square root of both sides. But wait, the square root of a square is the absolute value, right? But in this case, since the left side is a square and the right side is zero, the absolute value part might not be necessary here. Let me think. If I take the square root of both sides, I get:

√[(x + 2)^2] = √0

Which simplifies to:
|x + 2| = 0

But the absolute value of something is zero only when that something is zero. So, |x + 2| = 0 implies that x + 2 = 0. Therefore, solving for x:

x + 2 = 0

Subtract 2 

### Save model to Google Drive

In [16]:
from google.colab import drive
import shutil

drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
# Define the path in your Google Drive where you want to save the model
# You can change 'my_qwen_model' to any name you prefer
DRIVE_SAVE_PATH = "/content/drive/MyDrive/my_qwen_model"

# Copy the entire output directory to Google Drive
shutil.copytree(OUTPUT_DIR, DRIVE_SAVE_PATH, dirs_exist_ok=True)

print(f"Model and tokenizer saved to Google Drive at: {DRIVE_SAVE_PATH}")

Model and tokenizer saved to Google Drive at: /content/drive/MyDrive/my_qwen_model


In [20]:
import os

print(f"Listing contents of {DRIVE_SAVE_PATH}:")
# Use os.listdir to get the list of files and directories
# and print them one by one for better readability
for item in os.listdir(DRIVE_SAVE_PATH):
    print(item)

Listing contents of /content/drive/MyDrive/my_qwen_model:
adapter_model.safetensors
tokenizer_config.json
training_args.bin
adapter_config.json
chat_template.jinja
README.md
tokenizer.json
checkpoint-250
checkpoint-200


In [19]:
messages = [
    {
        "role": "user",
        "content": "Solve (x+3)^3 = 8."
    }
]

prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    output = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        pad_token_id=tokenizer.eos_token_id,
    )

answer = output[0][inputs["input_ids"].shape[1]:]
print(tokenizer.decode(answer, skip_special_tokens=True))

To solve the equation $(x + 3)^3 = 8$, follow these steps:

---

### **Step 1: Take the cube root of both sides**

We start by taking the cube root of both sides to eliminate the exponent:

$$
\sqrt[3]{(x + 3)^3} = \sqrt[3]{8}
$$

$$
x + 3 = \sqrt[3]{8}
$$

$$
x + 3 = 2
$$

---

### **Step 2: Solve for $x$**

Now, subtract 3 from both sides:

$$
x = 2 - 3
$$

$$
x = -1
$$

---

### **Final Answer:**

$$
\boxed{-1}
$$


### Load Model from Google Drive for Inference

In [23]:
# The drive should already be mounted from the previous step, but we can double check.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Define the path to the saved model in Google Drive
DRIVE_LOAD_PATH = "/content/drive/MyDrive/my_qwen_model"

# Re-initialize BitsAndBytesConfig as the model was trained with 4-bit quantization
bnb_config_load = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# Load the tokenizer
loaded_tokenizer = AutoTokenizer.from_pretrained(DRIVE_LOAD_PATH)

# Load the model with the quantization config
loaded_model = AutoModelForCausalLM.from_pretrained(
    DRIVE_LOAD_PATH,
    quantization_config=bnb_config_load,
    torch_dtype=torch.float16,
    device_map={"": 0},
)

loaded_model.eval()

print(f"Model and tokenizer loaded from: {DRIVE_LOAD_PATH}")

Mounted at /content/drive


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

Model and tokenizer loaded from: /content/drive/MyDrive/my_qwen_model


#### Example Inference with Loaded Model

In [27]:
inference_messages = [
    {
        "role": "user",
        "content": " Solve 4+8/7?"
    }
]

inference_prompt = loaded_tokenizer.apply_chat_template(
    inference_messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=False,
)

inference_inputs = loaded_tokenizer(inference_prompt, return_tensors="pt").to("cuda")

with torch.inference_mode():
    inference_output = loaded_model.generate(
        **inference_inputs,
        max_new_tokens=1020,
        do_sample=True,
        temperature=0.7,
        top_p=0.8,
        top_k=20,
        pad_token_id=loaded_tokenizer.eos_token_id,
    )

inference_answer = inference_output[0][inference_inputs["input_ids"].shape[1]:]
print(loaded_tokenizer.decode(inference_answer, skip_special_tokens=True))

To solve the expression \(4 + \frac{8}{7}\), follow these steps:

1. **Convert the whole number to a fraction with the same denominator as the fraction:**
   \[
   4 = \frac{28}{7}
   \]

2. **Add the fractions:**
   \[
   \frac{28}{7} + \frac{8}{7} = \frac{36}{7}
   \]

3. **Convert the improper fraction to a mixed number (optional):**
   \[
   \frac{36}{7} = 5 \frac{1}{7}
   \]

**Final Answer:**
\[
\boxed{5 \frac{1}{7}}
\]
